# Lesson 03 - Agentic Design Patterns

I denna lektion utforskar vi tre grundläggande designmönster för att bygga effektiva AI-agenter:

1. **Klart angivna agentinstruktioner** — Skapa precisa, roll-definierande uppmaningar som styr agentens beteende  
2. **Strukturerad output med Pydantic-modeller** — Säkerställa att agenter returnerar förutsägbar och validerad data  
3. **Agenter med enskilt ansvar** — Designa fokuserade agenter som var och en gör en sak väl

Vi kommer att tillämpa varje mönster på ett **rekommendationssystem för resmål**, och stegvis bygga ett system som kan föreslå destinationer, kontrollera tillgänglighet och hantera logistik.


## Installation


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Mönster 1: Tydliga agentinstruktioner

Det mest effektfulla mönstret är också det enklaste: att skriva tydliga, detaljerade instruktioner för din agent.

Bra instruktioner definierar:
- **Vem** agenten är (persona och ton)
- **Vad** den ska göra (ansvarsområden steg för steg)
- **Hur** den ska bete sig (begränsningar och stil)

Nedan skapar vi en reseconcierge-agent med uttryckliga instruktioner som formar varje svar den producerar.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Strukturerad utdata med Pydantic-modeller

Fri text är användbar för konversation, men system längre ner i kedjan behöver strukturerad data.
Genom att kombinera **Pydantic-modeller** med en **verktygsfunktion** kan vi:

- Definiera ett exakt schema för agentens utdata
- Validera svar automatiskt
- Integrera agentresultat i applikationslogik på ett pålitligt sätt

Vi introducerar också ett verktyg som returnerar destinationsdetaljer så att agenten grundar sina rekommendationer på verklig data.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Mönster 3: Agenter med Enkel Ansvarighet

Komplexa uppgifter gynnas av att fördela arbetet på flera fokuserade agenter, var och en med ett enda ansvar:

- En **Destinationsexpert** som känner till platser och tillgänglighet
- En **Logistikplanerare** som hanterar flyg, hotell och resplaner

Detta speglar principen för *separation av ansvar* inom mjukvaruteknik — varje agent är lättare att testa, underhålla och förbättra oberoende av varandra.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Sammanfattning

I denna lektion tillämpade vi tre agentiska designmönster på ett scenario för rese-rekommendationer:

| Mönster | Huvudidé | Fördel |
|---|---|---|
| **Klart definierade instruktioner** | Definiera persona, ansvar och begränsningar från början | Konsekvent agentbeteende i linje med varumärket |
| **Strukturerad output** | Använd Pydantic-modeller som svaretformat | Validerade, maskinläsbara resultat |
| **Enkelansvar** | Ge varje agent ett fokuserat uppdrag | Lättare att testa, underhålla och komponera |

Dessa mönster fungerar naturligt tillsammans — du kan kombinera klara instruktioner med strukturerad output inne i en enkelansvars-agent för att bygga robusta, produktionsfärdiga system.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, vänligen var medveten om att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För viktig information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som kan uppstå från användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
